# 09 — Build an Incident Card

**Target:** Apache Spark 4.2.0; Python 3.10+.

Run cells in order. These labs use small local workloads and avoid deliberately exhausting cluster resources. Read each *Why* before changing the experiment.

## Purpose
Make a one-page snapshot that lets another engineer identify the process and resource boundary before debugging.

**Why:** incidents are not the time to discover that the dashboard tracked the driver while an executor container hit its PID ceiling.

In [ ]:
from pyspark.sql import SparkSession
import os, platform, sys
spark=SparkSession.builder.master('local[2]').appName('thread-lab-09').getOrCreate()
sc=spark.sparkContext
card={
 'spark_version':spark.version,'python_version':sys.version.split()[0],
 'java_version':spark._jvm.java.lang.System.getProperty('java.version'),
 'os':platform.platform(),'driver_pid':os.getpid(),
 'master':sc.master,'application_id':sc.applicationId,
 'spark.executor.cores':sc.getConf().get('spark.executor.cores','<unset>'),
 'spark.executor.memory':sc.getConf().get('spark.executor.memory','<unset>'),
 'spark.executor.memoryOverhead':sc.getConf().get('spark.executor.memoryOverhead','<unset>'),
 'spark.speculation':sc.getConf().get('spark.speculation','<unset>'),
}
card

## Add deployment-specific fields
Fill in cluster manager, Spark Classic/Connect, executor lost reason, cgroup path/limits, worker process RSS/PSS, NMT startup setting, GC logs, event-log location, and whether speculation/retries occurred. Record timestamps and exact error text.

**Expected:** the card separates facts from hypotheses and identifies missing telemetry.

**Why:** values such as `memoryOverhead` have deployment-specific interpretation. Never paste credentials, tokens, or sensitive data into a notebook output.

In [ ]:
spark.stop()